# 6.0 - Meta-learning: Reptile vs first-order MAML

Both methods meta-learn a head for fast few-shot adaptation, then are probed on held-out classes at K in {1, 5, 10} shots. Reptile moves the meta-weights toward each episode's adapted weights; first-order MAML optimises the post-adaptation query loss. Everything runs on the cached features: base + task1 are the meta-train classes, task2 is held out.

## **Connect Colab**

In [1]:
import os, sys, subprocess
from getpass import getpass

os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

REPO   = "silvergjeka22/Unified-Lifelong-Action-Learning"
BRANCH = "embeddings"
ROOT   = "/content/ulal"

token = os.environ.get("GITHUB_TOKEN") or getpass("GitHub token: ")
os.environ["GITHUB_TOKEN"] = token

if os.path.isdir(os.path.join(ROOT, ".git")):
    subprocess.run(["git", "-C", ROOT, "fetch", "--depth", "1", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", ROOT, "reset", "--hard", "FETCH_HEAD"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH,
                    "https://" + token + "@github.com/" + REPO + ".git", ROOT], check=True)

if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

In [ ]:
os.environ["KAGGLE_USERNAME"] = "colab"
os.environ["KAGGLE_KEY"]      = "PASTE_YOUR_KAGGLE_KEY"

## **Download DATASET**

Drive only - the cached features hold everything, no video is extracted here.

In [ ]:
from src.bootstrap import setup
setup(drive=True, dataset=False)

In [4]:
# imports
%run /content/ulal/src/imports.py
set_seed(cfg.SEED)

ULAL_ROOT : /content/ulal
Device    : cuda
GPU       : Tesla T4 (15.6 GB)
Classes   : base=10 t1=10 t2=10 t3=10 t4=10


42

## **Load Cache**

Meta-train on base + task1 (classes 0-19); hold out task2 (classes 20-29) for the few-shot probe.

In [ ]:
CACHE, LSTM_STATE, META = load_feature_cache(cfg.FEAT_CACHE)
ALL_CLASSES_LIST = cfg.SELECTED_CLASSES + cfg.TASK_1 + cfg.TASK_2
check_cache_labels(META, ALL_CLASSES_LIST)

train_feats = torch.cat([CACHE["t0_train"][0], CACHE["t1_train"][0]])
train_labels = torch.cat([CACHE["t0_train"][1], CACHE["t1_train"][1]])
test_feats, test_labels = CACHE["t2_train"]

meta_train_classes = list(range(0, 20))
meta_test_classes = list(range(20, 30))
print(f"meta-train {tuple(train_feats.shape)} over {len(meta_train_classes)} classes")
print(f"meta-test  {tuple(test_feats.shape)} over {len(meta_test_classes)} classes")

## **Settings**

In [ ]:
N_WAY = 5
K_SHOT = 5
K_QUERY = 5
INNER_LR = 0.01
INNER_STEPS = 5
META_EPOCHS = 5
EPISODES = 50
EPSILON = 0.1
META_LR = 1e-3

## **Reptile meta-training**

In [ ]:
set_seed(cfg.SEED)
reptile_head = make_temporal_head(N_WAY, lstm_state=LSTM_STATE, device=device)
reptile_head = train_reptile(reptile_head, train_feats, train_labels, meta_train_classes,
                             N_WAY, K_SHOT, META_EPOCHS, EPISODES, INNER_LR, INNER_STEPS, EPSILON, device)

## **MAML meta-training (first-order)**

In [ ]:
set_seed(cfg.SEED)
maml_head = make_temporal_head(N_WAY, lstm_state=LSTM_STATE, device=device)
maml_head = train_maml(maml_head, train_feats, train_labels, meta_train_classes,
                       N_WAY, K_SHOT, K_QUERY, META_EPOCHS, EPISODES, INNER_LR, INNER_STEPS, META_LR, device)

## **Few-shot evaluation**

Adapt each meta-init on K support clips from the held-out classes and measure query accuracy.

In [ ]:
K_LIST = [1, 5, 10]
EVAL_EPISODES = 30
EVAL_QUERY = 15
rows = []
for K in K_LIST:
    set_seed(cfg.SEED)
    r_acc = few_shot_eval(reptile_head, test_feats, test_labels, meta_test_classes,
                          N_WAY, K, EVAL_QUERY, INNER_LR, INNER_STEPS, EVAL_EPISODES, device)
    set_seed(cfg.SEED)
    m_acc = few_shot_eval(maml_head, test_feats, test_labels, meta_test_classes,
                          N_WAY, K, EVAL_QUERY, INNER_LR, INNER_STEPS, EVAL_EPISODES, device)
    rows.append({"K": K, "reptile": r_acc, "maml": m_acc})
    print(f"K={K}: reptile {r_acc:.2%} | maml {m_acc:.2%}")

## **Compare**

In [ ]:
meta_df = pd.DataFrame(rows)
print(meta_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(meta_df["K"], meta_df["reptile"] * 100, marker="o", label="Reptile")
ax.plot(meta_df["K"], meta_df["maml"] * 100, marker="s", label="first-order MAML")
ax.set_xlabel("shots per class (K)")
ax.set_ylabel("query accuracy (%)")
ax.set_title(f"{N_WAY}-way few-shot: Reptile vs MAML on held-out classes")
ax.legend()
plt.tight_layout()
plt.savefig(f"{cfg.RESULTS_DIR}/stage_meta_reptile_vs_maml.png", dpi=150, bbox_inches="tight")
plt.show()

## **Save**

In [ ]:
payload = {
    "stage": "meta_reptile_vs_maml",
    "n_way": N_WAY,
    "meta_test_classes": [ALL_CLASSES_LIST[i] for i in meta_test_classes],
    "results": rows,
}
os.makedirs(cfg.RESULTS_DIR, exist_ok=True)
with open(f"{cfg.RESULTS_DIR}/stage_meta.json", "w") as f:
    json.dump(payload, f, indent=2, default=float)
print(f"Saved -> {cfg.RESULTS_DIR}/stage_meta.json")